# 07 · The fallback chain: when the AI is slow, limited or gone

**NASA Space Apps Cairo 2026 · Bootcamp · API & Data Integration**

**Goal:** an `ask_with_fallback()` that never crashes and never shows a blank screen: cache, then provider 1, provider 2, provider 3, then your own template.

**Done means:** you can classify an error as *retry* or *skip* · a 429 is retried politely · the chain moves to the next provider · if every provider fails the user still sees a clear, honest answer · good answers are cached.

> ✍️ **Starter notebook.** Cells marked `TODO` are yours to finish. Green ✅ check cells tell you when a step works.

Open in Colab: *File → Save a copy in Drive* first, so your edits are kept.

## Why a fallback matters
Free tiers run out in the middle of a demo. Models get renamed. Keys expire. Servers have bad minutes. Judges and users do not care why: they see a spinner or an error. You decide what they see instead.

| What happened | Signal | What your app does |
|---|---|---|
| Rate limit | `429` (maybe a `Retry-After` header) | wait a few seconds, **retry once**, then next provider |
| Provider trouble | `500` `502` `503` `504`, timeout, network error | retry once, then next provider |
| Bad key | `401` `403` | **skip** this provider (retrying cannot help) |
| Model gone | `404` | **skip** (fix the model name later) |
| Bad request | `400` | **skip** (do not hammer; fix your request) |
| Empty answer, `badshape` | `200` but nothing usable | **skip** |
| Everything failed | | show your **template** and say it is the basic version |

The last row is the same idea as the `backup/` file in the NASA notebooks: your app always has something honest to show.

In [ ]:
# ▶ Run this cell first
import os, json, re, time, hashlib, requests
import pandas as pd

def get_key(name):
    """Find a key in Colab Secrets (the 🔑 icon on the left), else in an environment variable.
    A key is never typed into a cell."""
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception:
        return os.environ.get(name) or None

def check(test, good, hint):
    """Tiny self-check: prints ✅ when the step works, ❌ plus a hint when it does not."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " + good) if ok else ("❌ " + hint))

# Every provider below speaks the same "chat completions" format, so ONE function can call all of them.
# Model names change often: if you get a 404, run the model list in notebook 05 and pick a current one.
PROVIDERS = {
    "gemini":     {"base": "https://generativelanguage.googleapis.com/v1beta/openai",
                   "key_name": "GEMINI_API_KEY", "model": "gemini-3.8-flash"},
    "groq":       {"base": "https://api.groq.com/openai/v1",
                   "key_name": "GROQ_API_KEY", "model": "llama-3.3-70b-versatile"},
    "openrouter": {"base": "https://openrouter.ai/api/v1",
                   "key_name": "OPENROUTER_API_KEY", "model": "openrouter/free"},
}
ORDER = ["gemini", "groq", "openrouter"]      # the order your app will try them in

def keys_found():
    return [n for n in ORDER if get_key(PROVIDERS[n]["key_name"])]

def redact(text):
    """Hide every key we know about, so messages are safe to show on a projector."""
    text = str(text)
    for p in PROVIDERS.values():
        k = get_key(p["key_name"])
        if k:
            text = text.replace(k, "YOUR_KEY")
    return text

def sample_banner():
    print("⚠️  SAMPLE MODE: this is illustrative text, not a real model answer. Add a free key to see real answers.")

print("Providers with a key:", keys_found() or "none yet. That is OK: the notebook still runs in sample mode.")

In [ ]:
SIMULATE = {}      # planned-failure switch, for example {"gemini": 429, "groq": 401}. Empty = real calls.

def raw_call(name, messages, want_json=False, max_tokens=600, temperature=0.3, timeout=30):
    """ONE attempt at ONE provider. Never raises. Returns a small dict you can inspect."""
    t0 = time.time()
    def fail(status, error, retry_after=None):
        return {"provider": name, "ok": False, "status": status, "error": redact(error)[:200],
                "retry_after": retry_after, "seconds": time.time() - t0}
    if name in SIMULATE:                                   # drill: pretend this provider failed
        return fail(SIMULATE[name], f"simulated {SIMULATE[name]}", 1 if SIMULATE[name] == 429 else None)
    cfg = PROVIDERS[name]
    key = get_key(cfg["key_name"])
    if not key:
        return fail("nokey", "no key set for " + name)
    body = {"model": cfg["model"], "messages": messages, "max_tokens": max_tokens, "temperature": temperature}
    if want_json:
        body["response_format"] = {"type": "json_object"}   # only a hint: always validate what comes back
    headers = {"Authorization": "Bearer " + key, "Content-Type": "application/json"}
    url = cfg["base"] + "/chat/completions"
    try:
        r = requests.post(url, headers=headers, json=body, timeout=timeout)
        if r.status_code == 400 and want_json:             # some models reject response_format: retry without it
            body.pop("response_format")
            r = requests.post(url, headers=headers, json=body, timeout=timeout)
    except requests.Timeout:
        return fail("timeout", "timed out")
    except requests.RequestException as e:
        return fail("network", type(e).__name__)
    if r.status_code != 200:
        try:
            wait = float(r.headers.get("Retry-After"))
        except (TypeError, ValueError):
            wait = None
        return fail(r.status_code, r.text, wait)
    try:
        data = r.json()
        choice = data["choices"][0]
        text = (choice["message"].get("content") or "").strip()
    except (ValueError, KeyError, IndexError, TypeError, AttributeError):
        return fail("badshape", "unexpected response shape")
    if not text:
        return fail("empty", "empty answer")
    return {"provider": name, "ok": True, "status": 200, "text": text,
            "finish_reason": choice.get("finish_reason"), "usage": data.get("usage", {}),
            "seconds": time.time() - t0}

`raw_call` is given: **one attempt at one provider, never raises**. Look at what it returns, then build the rest. It also has a **planned-failure switch**: set `SIMULATE = {"gemini": 429}` and `gemini` pretends to answer 429. You can rehearse every failure without burning your quota.

In [ ]:
MESSAGES = [
    {"role": "system", "content": "You explain space to a curious 12-year-old in two short sentences."},
    {"role": "user", "content": "Why do astronomers keep watching asteroids that will miss us?"},
]
TEMPLATE = "Astronomers track asteroids early so that, if one ever needed attention, there would be many years to prepare. (Basic version: the AI helper is not available right now.)"

print("Keys found for:", keys_found() or "no provider (the chain will end on your template, which is fine)")
first = raw_call(ORDER[0], MESSAGES)
print({k: first.get(k) for k in ("provider", "ok", "status", "error")})

## 1 · Retry or skip?
Write `action_for(status)`. Statuses are numbers (`429`) or words (`"timeout"`, `"network"`, `"nokey"`, `"empty"`, `"badshape"`).

### Your turn

In [ ]:
def action_for(status):
    # TODO: return "retry" for 429, 500, 502, 503, 504, "timeout" and "network"; "skip" for everything else
    ...

In [ ]:
check(lambda: all(action_for(s) == "retry" for s in (429, 503, "timeout")) and all(action_for(s) == "skip" for s in (400, 401, 403, 404, "nokey", "empty")),
      "Retry/skip rules are right.", "Retry: 429, 5xx, timeout, network. Skip: 400, 401, 403, 404, nokey, empty, badshape.")

## 2 · Retry politely
`call_with_retry(name, messages, max_retries=1)` calls `raw_call`. If it fails with a *retry* status it waits (use the `retry_after` the provider sent, otherwise 2 seconds, **never more than 10**) and tries again, at most `max_retries` times. A *skip* status stops immediately. Return `(last_result, list_of_all_attempts)`.

### Your turn

In [ ]:
def call_with_retry(name, messages, max_retries=1, **kw):
    attempts = []
    for i in range(max_retries + 1):
        res = raw_call(name, messages, **kw)
        attempts.append(res)
        # TODO: stop the loop if res["ok"] or the action is "skip"
        # TODO: otherwise, if there is a retry left, sleep min(res.get("retry_after") or 2, 10) seconds
        ...
    return res, attempts

In [ ]:
def _t():
    SIMULATE.clear(); SIMULATE["gemini"] = 429
    r1, a1 = call_with_retry("gemini", MESSAGES)
    SIMULATE["gemini"] = 401
    r2, a2 = call_with_retry("gemini", MESSAGES)
    SIMULATE.clear()
    return len(a1) == 2 and not r1["ok"] and len(a2) == 1
check(_t, "429 is tried twice, 401 only once.", "A 429 should give 2 attempts (one retry); a 401 should give 1 attempt.")

## 3 · Remember good answers (a cache)
The same question twice should cost nothing. A tiny cache also means your demo still works if every provider is down *after* it worked once: the same idea as saving a NeoWs copy.

`cache_key` makes a short id from the messages; `cache_get` and `cache_put` read and write `backup/llm_cache.json`.

### Your turn

In [ ]:
CACHE_FILE = "backup/llm_cache.json"

def cache_key(messages):
    # TODO: a short id: sha256 of json.dumps(messages, sort_keys=True).encode(), first 16 hex characters
    ...

def cache_get(k):
    # TODO: read CACHE_FILE (a JSON dict) and return the value for k, or None if the file is missing or broken
    ...

def cache_put(k, text):
    os.makedirs("backup", exist_ok=True)
    # TODO: load the existing dict (or start with {}), set data[k] = text, write it back
    ...

In [ ]:
def _t():
    k = cache_key([{"role": "user", "content": "cache test"}])
    cache_put(k, "hello")
    return len(k) == 16 and cache_get(k) == "hello" and cache_get("nope") is None
check(_t, "Cache saves and finds answers.", "cache_key must be 16 characters; cache_put then cache_get must return the same text.")

## 4 · The chain
`ask_with_fallback(messages, fallback_text, order=None, use_cache=True, **kw)`:
1. If `use_cache` and the answer is cached → return it with `provider: "cache"`.
2. For each provider in `order` (default `ORDER`): `call_with_retry`. Add a row `{provider, status, seconds}` to a `log` for every attempt. If it worked: cache the text and return `{text, provider, attempts: log}`.
3. If every provider failed: print a one-line warning and return your **template** with `provider: "template"`.

The user must always be able to tell that it is the basic version. Never present a template as an AI answer.

### Your turn

In [ ]:
def ask_with_fallback(messages, fallback_text, order=None, use_cache=True, **kw):
    k = cache_key(messages)
    # TODO 1: if use_cache and cache_get(k): return {"text": ..., "provider": "cache", "attempts": []}
    log = []
    for name in (order or ORDER):
        res, tries = call_with_retry(name, messages, **kw)
        log += [{"provider": t["provider"], "status": t["status"], "seconds": round(t["seconds"], 2)} for t in tries]
        # TODO 2: if res["ok"]: cache_put(k, res["text"]) and return {"text", "provider": name, "attempts": log}
        ...
    # TODO 3: print a one-line warning and return the template with provider "template"
    ...

In [ ]:
def _t():
    SIMULATE.update({n: 503 for n in ORDER})
    r = ask_with_fallback(MESSAGES, TEMPLATE, use_cache=False, max_retries=0)
    SIMULATE.clear()
    return r["provider"] == "template" and r["text"] == TEMPLATE and len(r["attempts"]) == len(ORDER)
check(_t, "When everything fails you return the template.", "With every provider failing (503) you must return the template and log one attempt per provider.")

## 5 · Rehearse the failures
Each drill breaks something on purpose and shows who answered. `use_cache=False` makes sure an old cached answer does not hide the failure. **If you have no key at all, every drill ends on the template. That is the point: the app still works.**

In [ ]:
def drill(title, simulate):
    SIMULATE.clear(); SIMULATE.update(simulate)
    res = ask_with_fallback(MESSAGES, TEMPLATE, use_cache=False, max_retries=1)
    SIMULATE.clear()
    print(f"\n=== {title}")
    print("answered by:", res["provider"])
    print(pd.DataFrame(res["attempts"]).to_string(index=False) if res["attempts"] else "(no attempts)")
    return res

r1 = drill("1. First provider is rate-limited (429)", {ORDER[0]: 429})
r2 = drill("2. First provider has a bad key (401), second is gone (404)", {ORDER[0]: 401, ORDER[1]: 404})
r3 = drill("3. Everything fails (503)", {n: 503 for n in ORDER})

What would you want to see in a post-mortem? Add one more drill: **a timeout on the first provider**, and print who answered.

In [ ]:
r4 = drill("4. First provider times out", ...)      # TODO: {ORDER[0]: "timeout"}

In [ ]:
check(lambda: r4["provider"] in ORDER[1:] + ["template", "cache"] and r4["attempts"][0]["status"] == "timeout",
      "Timeout handled.", "Simulate a 'timeout' on ORDER[0] and check the first attempt in the log says timeout.")

## 6 · Choose your order and your template
Think, then change `ORDER` at the top if you like:
- Put the provider with the **most generous free limits** and best uptime first.
- Keep a **different company** second, so one outage cannot take both.
- Write a **template** that is true and useful without AI (numbers from your data, plain words). A good template is the difference between "the app is broken" and "the app is working, just simpler".

Also: an LLM call can take seconds. Use a `timeout`, never call inside a tight loop, and cache.

## ✅ Done means
- [ ] `action_for` tells retry from skip
- [ ] a 429 gets one polite retry (never a long sleep)
- [ ] the chain moves on to the next provider and logs every attempt
- [ ] with all providers failing, the user sees an honest template
- [ ] good answers are cached

**Next:** notebook `08`: put an LLM into a feature that uses NASA data.